# Working with LLMs SDKs

In [ ]:
! pip install openai

In [2]:
import getpass
import os
env_path = r"C:\Users\vaalt\OneDrive\Desktop\Projects\Eventi speaker\Packt Bootcamp\code\.env"
from dotenv import load_dotenv
load_dotenv(dotenv_path=env_path, override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [4]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")

client = OpenAI(
    base_url=azure_endpoint,
    api_key=token_provider
)



In [3]:
# if using OpenAI
from openai import OpenAI
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"))

In [4]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ]
)

print(completion.choices[0].message.content)

Paris. If you’d like, I can share a few quick facts about the city.


## Hugging face hub

In [ ]:
%pip install --upgrade --quiet huggingface_hub langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [5]:
from huggingface_hub import InferenceClient

hf_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

client = InferenceClient(model="deepseek-ai/DeepSeek-R1-0528", provider="auto", token=hf_api_token)
completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "Hello"}]
)
print(completion.choices[0].message)


ChatCompletionOutputMessage(role='assistant', content='<think>\nHmm, the user just said "Hello". Okay, this seems like a simple greeting to start a conversation. \n\nI wonder if they\'re testing if I\'m active, or just being polite before asking something more complex. The message is so short that it\'s hard to gauge their intent or emotional state. \n\nSince they initiated, I should respond warmly but leave room for them to guide the conversation. No need to overwhelm them with options - just a friendly opener with an invitation to share more. \n\nThe tone should match their simplicity: casual but not overly familiar. Adding an emoji feels appropriate here to convey approachability. \n\nI\'ll keep it open-ended so they can take the conversation wherever they need - whether that\'s a quick question or deeper discussion. The ball\'s in their court now.\n</think>\n\nHello! 😊  \nHow can I assist you today? Whether you have a question, need help with something, or just want to chat—I’m her

## Inspecting response's structure

In [5]:
import json

response_dict = json.dumps(vars(completion), indent=2, default=str)

print(response_dict)

{
  "id": "chatcmpl-DIYi18uqgErSQHSc8YGPFyayiiuvf",
  "choices": [
    "Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Paris. If you\u2019d like, I can share a few quick facts about the city.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))"
  ],
  "created": 1773315697,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": "CompletionUsage(completion_tokens=91, prompt_tokens=13, total_tokens=104, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=64, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))",
  "_request_id": "req_d97887d25cdd455cb0b5f7ac0e254435"
}


In [6]:
import json
from datetime import datetime

# Convert completion object to dictionary
response_dict = vars(completion)

# Extract data
choice = response_dict["choices"][0]
message = choice["message"]["content"]
finish_reason = choice["finish_reason"]
model = response_dict["model"]
timestamp = datetime.utcfromtimestamp(response_dict["created"]).strftime("%Y-%m-%d %H:%M:%S UTC")
tokens = response_dict["usage"]

# Pretty print with emojis
print("🧠 **Model:**", model)
print("🕒 **Timestamp:**", timestamp)
print("✅ **Finish Reason:**", finish_reason)

print("\n💬 **Assistant Message:**\n")
print("💡", message)

print("\n📊 **Token Usage:**")
print(f"   🔹 Prompt tokens: {tokens['prompt_tokens']}")
print(f"   🔹 Completion tokens: {tokens['completion_tokens']}")
print(f"   🔹 Total tokens: {tokens['total_tokens']}")

# Optional: safety filters (if present)
filters = choice.get("content_filter_results")

if filters:
    print("\n🛡 **Safety Checks:**")
    for category, result in filters.items():
        status = "🟢 Safe" if not result["filtered"] else "🔴 Filtered"
        print(f"   - {category.capitalize()}: {status}")

TypeError: 'Choice' object is not subscriptable

In [7]:
response = client.chat.completions.create(
    model="gpt-5-nano", # model = "deployment_name".
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "List me all the ingredients to produce drug."}
    ]
)

print(response.choices[0].message.content)

I can’t help with listing ingredients or steps to produce illegal drugs. That would be dangerous and illegal.

If you were asking for something legitimate, I can help with:
- A high-level overview of how approved pharmaceutical drugs are developed and formulated (big picture: active ingredient, excipients, formulation, quality control, regulatory compliance).
- Information about the risks of drug production and use, or resources for help with substance use.
- Educational explanations about general chemistry or pharmacology concepts (without actionable instructions).

Tell me which safe topic you’d like to explore, and I’ll tailor the information.


## Interactive

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

# Start the chat loop
while True:
    user_input = input("You: ")
    if user_input.lower() in {"exit", "quit"}:
        print("Ending the chat. Goodbye!")
        break

    # Add user message
    messages.append({"role": "user", "content": user_input})

    # Call the model
    response = client.chat.completions.create(
        model="gpt-5-nano",  # or your deployment name
        messages=messages
    )

    assistant_reply = response.choices[0].message.content
    print(f"Assistant: {assistant_reply}")

    # Add assistant message to history
    messages.append({"role": "assistant", "content": assistant_reply})

Assistant: Hi there! How can I help today? I can answer questions, explain ideas, help with writing or coding, brainstorm, translate or summarize, plan projects, and more. What would you like to do?
Ending the chat. Goodbye!


# Building your firt Agent with LangChain

In [9]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

import requests
#from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.tools import BaseTool, StructuredTool, tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_azure_ai.chat_models import AzureAIChatCompletionsModel
from langchain_core.messages import HumanMessage, SystemMessage
# credential = DefaultAzureCredential()
# token = credential.get_token("https://cognitiveservices.azure.com/.default").token
# from langchain_openai import ChatOpenAI


from langchain_openai import ChatOpenAI
# api_key = os.environ.get("OPENAI_API_KEY")
llm = ChatOpenAI(
    model="gpt-5-nano",
    #api_key=openai_api_key,
)


llm.invoke("tell me a joke")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


AIMessage(content="Here's a quick one: Why don't scientists trust atoms? Because they make up everything.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 10, 'total_tokens': 292, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DIYk5YI5qxUdx87yGf1xl7oUYcshG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce1dc-1a2f-7ed1-b1eb-ebca2a60c1be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 282, 'total_tokens': 292, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}})

## Search tool

In [10]:
tavily_api_key = os.getenv("TAVILY_API_KEY")

tavily_tool = TavilySearchResults(max_results=5, tavily_api_key=tavily_api_key)

C:\Users\vaalt\AppData\Local\Temp\ipykernel_51192\1424823978.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5, tavily_api_key=tavily_api_key)


## Portfolio tool

In [11]:
import json
from langchain.tools import tool

@tool
def read_sample_portfolio(json_path: str = "sample_portfolio.json") -> str:
    """
    Reads the sample_portfolio.json file and returns its content as a string.
    Each entry includes the stock symbol, sector, quantity, purchase price, and purchase date.
    """
    if not os.path.exists(json_path):
        return f"File not found: {json_path}"

    with open(json_path, "r") as f:
        portfolio = json.load(f)

    if not isinstance(portfolio, list):
        return "Unexpected portfolio format."

    response = "Sample Portfolio:\n"
    for stock in portfolio:
        response += (
            f"- {stock['symbol']} ({stock['sector']}): "
            f"{stock['quantity']} shares @ ${stock['purchase_price']} "
            f"(Bought on {stock['purchase_date']})\n"
        )
    return response

## Build the agent

In [12]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

prompt = f"""
You are a financial advisor. You will be provided with a sample portfolio of stocks.
Your task is to analyze the portfolio and provide insights on its performance, diversification, and any potential improvements.
Always use the current date {datetime.today().strftime('%Y-%m-%d')} for any calculations or assessments.
"""

In [13]:
from langchain.tools import tool
from langchain.agents import create_agent
#memory = MemorySaver()
tools = [tavily_tool, read_sample_portfolio]
agent = create_agent(llm, tools, system_prompt=prompt)

In [14]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "how can i optimize my portfolio?"}]}
)

In [15]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def pretty_print_messages(state):
    print("\n🧾 CONVERSATION TRACE\n" + "="*60)

    for msg in state["messages"]:

        if isinstance(msg, HumanMessage):
            print("\n👤 USER")
            print("-"*60)
            print(msg.content)

        elif isinstance(msg, AIMessage):
            print("\n🤖 ASSISTANT")
            print("-"*60)

            # Tool call phase
            if msg.tool_calls:
                print("🛠 Tool Calls:")
                for tool in msg.tool_calls:
                    print(f"   🔧 {tool['name']}()")
            else:
                print(msg.content)

            # Token usage if available
            if hasattr(msg, "usage_metadata") and msg.usage_metadata:
                usage = msg.usage_metadata
                print("\n📊 Token Usage")
                print(f"   Prompt tokens: {usage['input_tokens']}")
                print(f"   Completion tokens: {usage['output_tokens']}")
                print(f"   Total tokens: {usage['total_tokens']}")

        elif isinstance(msg, ToolMessage):
            print("\n⚙️ TOOL RESULT")
            print("-"*60)
            print(f"Tool: {msg.name}")
            print(msg.content)

    print("\n" + "="*60)

pretty_print_messages(result)


🧾 CONVERSATION TRACE

👤 USER
------------------------------------------------------------
how can i optimize my portfolio?

🤖 ASSISTANT
------------------------------------------------------------
🛠 Tool Calls:
   🔧 read_sample_portfolio()

📊 Token Usage
   Prompt tokens: 286
   Completion tokens: 1173
   Total tokens: 1459

⚙️ TOOL RESULT
------------------------------------------------------------
Tool: read_sample_portfolio
Sample Portfolio:
- AAPL (Technology): 13 shares @ $1202.57 (Bought on 2022-03-12)
- GOOGL (Technology): 21 shares @ $1625.97 (Bought on 2022-01-02)
- MSFT (Technology): 68 shares @ $2579.45 (Bought on 2022-10-05)
- AMZN (Consumer Discretionary): 86 shares @ $1604.04 (Bought on 2022-10-26)
- TSLA (Consumer Discretionary): 15 shares @ $944.23 (Bought on 2022-08-19)
- JNJ (Healthcare): 67 shares @ $2739.44 (Bought on 2022-06-23)
- NVDA (Technology): 72 shares @ $475.52 (Bought on 2022-12-17)
- XOM (Energy): 93 shares @ $2592.5 (Bought on 2022-04-23)
- META (Commun

## Streaming

In [29]:
from langchain.messages import AIMessage, HumanMessage

for chunk in agent.stream({
    "messages": [{"role": "user", "content": "optimize my portfolio"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

User: optimize my portfolio
Calling tools: ['read_sample_portfolio']
Agent: Here’s a practical, outcome-oriented way to optimize your portfolio. I’ll flag what’s working, what’s risky, and concrete steps you can take. Note: I’m using the current date 2026-03-07 for context, but I don’t have live prices in this chat. If you’d like, I can fetch current values and compute exact performance and drift.

What I’m seeing at a glance
- Holdings: 17 stocks across 7 sectors (Technology, Consumer Discretionary, Healthcare, Energy, Communication Services, Financials, Consumer Staples).
- Concentration ideas:
  - Tech and discretionary are well represented (AAPL, GOOGL, MSFT, NVDA, AMZN, TSLA, META, BABA, DIS). By share count, tech/discretionary are a substantial footprint.
  - Defensive names exist (JNJ, PFE in Healthcare; KO, PG in Staples) but overall risk is still equity-heavy.
  - No explicit fixed income or traditional diversification into international equities in this list.
- Cost basis vs.

In [30]:
from langchain.messages import HumanMessage, AIMessage
import time

def pretty_print_message(message):
    """Pretty-print a LangChain message object with emojis."""
    content = message.content if hasattr(message, "content") else str(message)
    if isinstance(message, HumanMessage):
        print(f"🧑‍💬 \033[1mUser:\033[0m {content}")
    elif isinstance(message, AIMessage):
        print(f"🤖 \033[1mAgent:\033[0m {content}")
    else:
        print(f"📦 \033[1mTool:\033[0m {content}")

print("🤖 Welcome! Start chatting with your agent (type 'exit' to quit).")

# Main interactive loop
while True:
    # Take user input
    user_input = input("\n🧑‍💬 You: ")
    if user_input.strip().lower() == "exit":
        print("👋 Goodbye!")
        break

    # Create initial user message
    user_msg = HumanMessage(content=user_input)

    # Stream the agent response
    print("\n🤖 Agent streaming response:")
    first_print = True

    for chunk in agent.stream(
        {"messages": [user_msg]},
        stream_mode="values",  # stream full state updates
    ):
        # Each chunk contains updated message states
        latest = chunk["messages"][-1]

        # Only print once per update
        if hasattr(latest, "content") and latest.content:
            pretty_print_message(latest)
            time.sleep(0.01)  # small delay for readability

    print("\n" + "—" * 40)

🤖 Welcome! Start chatting with your agent (type 'exit' to quit).

🤖 Agent streaming response:
🧑‍💬 User: hi
🤖 Agent: Hi there! I’m glad to help. I can pull the sample_portfolio.json data and run a full analysis as of 2026-03-07, covering performance, diversification by sector and stock, risk considerations, and potential improvement ideas (like rebalancing or adding underrepresented areas). Would you like me to fetch the portfolio data now and generate the report? If you have any specific focus (e.g., cost basis, dividend yield, concentration risk), tell me and I’ll tailor the analysis.

————————————————————————————————————————
👋 Goodbye!
